# USER PARAMETERS

This notebook builds the mixed cusp–ray geometry, transports it to $m=s+r$ half-line copies, and explains how each ordered branch pair selects a Cauchy block class. Edit the mathematical and animation values in the next cell, then run all cells from a clean kernel. The exported GIF, MP4, PNG and vector PDFs are saved in `../outputs/`.

In [1]:
# ============================================================
# USER PARAMETERS
# ============================================================

# MATHEMATICAL PARAMETERS
import math
s = 2
r = 2
m = s + r
# Computed lists adapt when s or r changes; replace the lists for custom values.
n_negative = (s + 1) // 2
n_positive = s - n_negative
c_values = ([-1.45 + (0.60*i/(n_negative-1) if n_negative > 1 else 0.0)
             for i in range(n_negative)]
            + [0.85 + (0.60*i/(n_positive-1) if n_positive > 1 else 0.0)
               for i in range(n_positive)])
beta_values = ([math.pi] if r == 1 else
               [0.60*math.pi + 0.80*math.pi*i/(r-1) for i in range(r)])
p = 2.0
delta = 0.0
X_PARAMETER_MIN = 1e-4
X_PARAMETER_MAX = 65.0
N_GEOMETRY_POINTS = 340

# ANIMATION PARAMETERS
FPS = 24
SPEED_FACTOR = 1.0               # >1 shortens every stage duration
HOLD_START = 0.3 / SPEED_FACTOR
DRAW_GEOMETRY_TIME = 1.2 / SPEED_FACTOR
HOLD_GEOMETRY = 0.3 / SPEED_FACTOR
TRANSPORT_TIME = 1.1 / SPEED_FACTOR
VECTOR_TIME = 0.7 / SPEED_FACTOR
MATRIX_DURATION = 1.1 / SPEED_FACTOR
BLOCK_HIGHLIGHT_DURATION = 0.7 / SPEED_FACTOR
FORMULA_HOLD_DURATION = 2.6 / SPEED_FACTOR  # exact/model/class; final class pause is 58%
HOLD_END = 1.2 / SPEED_FACTOR
# Named aliases used in the storyboard specification:
BUILD_MATRIX_TIME = MATRIX_DURATION
BLOCK_TIME = BLOCK_HIGHLIGHT_DURATION
FORMULA_TIME = FORMULA_HOLD_DURATION
FINAL_HOLD = HOLD_END
OUTPUT_NAME = "branch_interactions_v1"
OUTPUT_GIF = f"{OUTPUT_NAME}.gif"
OUTPUT_MP4 = f"{OUTPUT_NAME}.mp4"
OUTPUT_PNG = f"{OUTPUT_NAME}.png"
OUTPUT_STATIC_PDF = "branch_interactions_static.pdf"
SHOW_EXACT_KERNEL = True
SHOW_COMPACT_MODEL = True
SHOW_OPERATOR_CLASS = True

# Branch interactions on a mixed cusp–ray star

**Question.** How does the geometry of the input branch $j$ and output branch $k$ determine a transported Cauchy block?

The animation follows three mathematical layers in order: the **exact transported operator**, the **model modulo compact operators**, and the **finite Fredholm symbol over $\widehat{\Omega}$**. The arrows from the curve to the half-lines are schematic representations of an isometric transport of functions, not a literal deformation of the curve.

In [2]:
from pathlib import Path
import sys
from IPython.display import Markdown, display

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
elif not (project_dir / "src" / "geometry.py").exists():
    project_dir = project_dir / "karlovich_2025_cusps"
if not (project_dir / "src" / "geometry.py").exists():
    raise FileNotFoundError("Open this notebook from its project folder or the repository root.")
sys.path.insert(0, str(project_dir / "src"))

from geometry import BranchParameters, default_c_values, default_beta_values, cell_counts
from interactions import count_by_class
from operator_metadata import INTERACTIONS, interaction_table_markdown
from render_branch_interactions import AnimationTiming, render_assets, render_frame

assert m == s + r
c_values = default_c_values(s) if c_values is None else tuple(c_values)
beta_values = default_beta_values(r) if beta_values is None else tuple(beta_values)
params = BranchParameters(s, r, tuple(c_values), tuple(beta_values), p, delta,
                          X_PARAMETER_MIN, X_PARAMETER_MAX, N_GEOMETRY_POINTS)
timing = AnimationTiming(FPS, HOLD_START, DRAW_GEOMETRY_TIME, HOLD_GEOMETRY,
                         TRANSPORT_TIME, VECTOR_TIME, MATRIX_DURATION,
                         BLOCK_HIGHLIGHT_DURATION, FORMULA_HOLD_DURATION, HOLD_END)
assert count_by_class(params) == {"CC": s*s, "CR": s*r, "RC": r*s, "RR": r*r}
print(f"Python: {sys.executable} ({sys.version.split()[0]})")
print(f"s={s}, r={r}, m={params.m}; c={c_values}; beta={[round(math.degrees(v), 1) for v in beta_values]} degrees")
print(f"FPS={FPS}; SPEED_FACTOR={SPEED_FACTOR}; planned duration={timing.duration:.2f} s")

Python: /home/enriquedo/.pyenv/versions/3.12.2/bin/python (3.12.2)
s=2, r=2, m=4; c=(-1.45, 0.85); beta=[108.0, 252.0] degrees
FPS=24; SPEED_FACTOR=1.0; planned duration=17.00 s


## Mathematical checkpoint before drawing

**Columns are input $j$; rows are output $k$.** The entry $(k,j)$ is the contribution of input $\varphi_j$ to output component $k$. Let $T_{k,j}=\Upsilon_k S_{\Gamma_j}\Upsilon_j^{-1}$. With $X=L^p(\mathbb R_+,w)$, the exact kernel is defined by

$$T_{k,j}\varphi_j(x)=\frac{1}{\pi i}\int_0^\infty K^{\mathrm{ex}}_{k,j}(x,\tau)\varphi_j(\tau)\,d\tau,$$

with principal value on the diagonal. In the compact model, $T_{k,j}-S_{k,j}\in\mathcal K(X)$; this is **not** an exact equality of kernels.

In [3]:
display(Markdown(interaction_table_markdown()))
print("Source: Karlovich 2025, geometry (1.1)–(1.2); exact blocks (3.6); compact models and classes (3.21)–(3.28), Lemmas 3.1–3.2, Corollary 3.3.")

| Interaction | input $j$ | output $k$ | exact $K^{\rm ex}_{k,j}$ | compact model $K^{\rm mod}_{k,j}$ | operator class | source |
|---|---|---|---|---|---|---|
| CC | $j\leq s$ | $k\leq s$ | $\frac{1}{\tau+i(c_k-c_j)-x}$ | $k=j:\ K^{\mathrm{ex}}_{k,j};\ k\ne j:\ \frac{\chi_\infty(x)\chi_\infty(\tau)}{\tau+i(c_k-c_j)-x}$ | diagonal Mellin; off-diagonal localized Wiener–Hopf | Karlovich 2025 exact (3.6); model (3.22), (3.24)-(3.28), Lemma 3.1, Corollary 3.3 |
| CR | $j>s$ | $k\leq s$ | $\frac{e^{i\beta_j}}{\tau+i c_k e^{i\beta_j}-e^{i\beta_j}x}$ | $\chi_\infty(x)\chi_\infty(\tau)\frac{e^{i\beta_j}}{\tau-e^{i\beta_j}x}$ | localized Mellin convolution | Karlovich 2025 exact (3.6); model (3.22), (3.24)-(3.28), Lemma 3.1, Corollary 3.3 |
| RC | $j\leq s$ | $k>s$ | $\frac{e^{-i\beta_k}}{\tau-i c_j-e^{-i\beta_k}x}$ | $\chi_\infty(x)\chi_\infty(\tau)\frac{e^{-i\beta_k}}{\tau-e^{-i\beta_k}x}$ | localized Mellin convolution | Karlovich 2025 exact (3.6); model (3.22), (3.24)-(3.28), Lemma 3.1, Corollary 3.3 |
| RR | $j>s$ | $k>s$ | $\frac{e^{i(\beta_j-\beta_k)}}{\tau-e^{i(\beta_j-\beta_k)}x}$ | $K^{\mathrm{ex}}_{k,j}$ | Mellin convolution | Karlovich 2025 exact (3.6); model (3.22), (3.24)-(3.28), Corollary 3.3 |

Source: Karlovich 2025, geometry (1.1)–(1.2); exact blocks (3.6); compact models and classes (3.21)–(3.28), Lemmas 3.1–3.2, Corollary 3.3.


## Build the animation

Stages 1–5 construct $\Gamma$, the $m$ half-line copies, the vector $\varphi$, and the $m\times m$ matrix from the indices. Stages 6–9 reveal CC, CR, RC, and RR one at a time, showing the exact kernel before the compact model. Stage 10 summarizes the three layers.

In [4]:
output_dir = project_dir / "outputs"
assets = render_assets(params, timing, output_dir, OUTPUT_NAME,
                       output_gif=OUTPUT_GIF, output_mp4=OUTPUT_MP4,
                       show_exact_kernel=SHOW_EXACT_KERNEL,
                       show_compact_model=SHOW_COMPACT_MODEL,
                       show_operator_class=SHOW_OPERATOR_CLASS)
assert assets["gif"].name == OUTPUT_GIF and assets["mp4"].name == OUTPUT_MP4
assert assets["png"].name == OUTPUT_PNG and assets["pdf"].name == OUTPUT_STATIC_PDF
for name in ("gif", "mp4", "png", "pdf", "pdf_v1", "beamer_pdf"):
    path = assets[name]
    print(f"{name:7s}: {path.relative_to(project_dir)} ({path.stat().st_size:,} bytes)")
print(f"Rendered {assets['frames']} frames; duration {assets['duration_seconds']:.2f} s.")

gif    : outputs/branch_interactions_v1.gif (1,375,029 bytes)
mp4    : outputs/branch_interactions_v1.mp4 (192,315 bytes)
png    : outputs/branch_interactions_v1.png (102,618 bytes)
pdf    : outputs/branch_interactions_static.pdf (20,780 bytes)
pdf_v1 : outputs/branch_interactions_static_v1.pdf (20,780 bytes)
beamer_pdf: outputs/branch_interactions_beamer.pdf (20,171 bytes)
Rendered 406 frames; duration 16.92 s.


## Interpret the result

- The geometry determines the **type of each ordered pair** $(k,j)$; it does not by itself prove compactness.
- The full transported entry is exact. Karlovich then separates compact remainders to obtain the Wiener–Hopf/Mellin model.
- The final determinant belongs to a finite symbol matrix over $\widehat{\Omega}$, after this reduction.

**Try next.** Set `s=3`, `r=2`, update `c_values` to `[-1.45, -0.85, 0.85]`, and rerun. The grid becomes $5\times5$ and the class counts become $9,6,6,4$ for CC, CR, RC, RR.

Executable source: [render_branch_interactions.py](../src/render_branch_interactions.py).

<!-- resultado-verificado: branch_math -->